# Step 2 · Taking snapshots inside the model

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ about 30 minutes · ⚡ GPU recommended

> ### 🤔 Wonder
> If we pause the model right after it reads a **pain** sentence, and again after a **comfort** sentence, are the paused numbers different in a *consistent* way?

**In this step you will:**

1. Meet the lab's sentences and learn why they're split into three groups.
2. Decide *where* in the model to take a snapshot, and why.
3. Build a **hook**, a clip-on sensor that copies numbers as they flow through a layer.
4. Collect snapshots for 44 sentences into one 3-D block of numbers, and describe its shape in words.

## Setup

Each notebook starts fresh, so we load the model again. (In Colab, each notebook runs in its own session.)

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

In [ ]:
# 🔧 Load the model: just run this cell (the first download is about 3 GB and takes a few minutes).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Keep this model for class. 'Qwen/Qwen2.5-0.5B' uses less memory, but its results
# will differ, so record it as a different experiment.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

SEED = pclab.set_seed(42)          # fixes random choices so results can be repeated
DEVICE = pclab.device_report()     # 'cuda' (GPU) or 'cpu'
tokenizer, model = pclab.load_model(MODEL_NAME)
LAYERS = pclab.get_layers(model)
N_LAYERS = len(LAYERS)
HIDDEN_SIZE = model.config.hidden_size
pclab.use_lab_style()
print(f"Loaded {pclab.model_facts(model)['model']}: {N_LAYERS} layers, {HIDDEN_SIZE} numbers per token")

## Part 1 · Our sentences, and why there are three groups

The sentences come in **pairs** that describe the same situation two ways:

> **Comfort:** The tea warmed her throat pleasantly.
> **Pain:** The tea burned her throat when she swallowed.

Pairing keeps the topic (tea, throat) the same, so the main difference *should* be comfort versus pain.

The pairs are split into three groups, like a course:

| Group | Like… | Size | What we use it for |
|---|---|---:|---|
| **Build** | practice problems | 16 pairs (32 sentences) | build our pain–comfort "ruler" (Step 3) |
| **Validation** | a practice quiz | 6 pairs (12 sentences) | choose which layer works best (Step 3) |
| **Test** | the final exam | 8 pairs (16 sentences) | report how well the ruler works; first opened in Step 4 and **never** used to make choices |

If we peeked at the final exam while making choices, our score would look better than it really is. That mistake is called **test leakage**. Avoiding it is why the test set stays sealed until Step 4.

We call this the "pain–comfort" lab, but the sentences are specifically about **physical pain versus physical comfort** (a hot bath, a scratchy scarf, a soft pillow), not happiness or emotion in general.

**✏️ Predict first:** comfort sentences use words like *soothed* and *soft*; pain sentences use *burned* and *sting*. Could a simple program that just counts words get this task right, without understanding anything about bodies? (You'll test this in Step 5.)

In [ ]:
# 🔧 Load the sentences from data/sentences.csv
sentences = pclab.load_sentences()
look(sentences, 'sentences')

> **🧱 Why a CSV file and a DataFrame?** The sentences live in a plain spreadsheet file (`data/sentences.csv`), not inside the code. Anyone can read, check, cite or extend them without touching Python. A DataFrame keeps each sentence next to its labels, so they can never get out of step.

### 📦 What does `groupby` return? Two steps, looked at separately

A common place to get lost is a chain like `sentences.groupby([...]).size().unstack()`. Split the chain and look after each link.

In [ ]:
counts = sentences.groupby(['split', 'category']).size()
look(counts, 'counts (after groupby + size)')
print()
print(counts)

In [ ]:
table = counts.unstack()
look(table, 'table (after unstack)')

### 🔍 Code walk-through

| Link | Returns | What it did |
|---|---|---|
| `.groupby(['split', 'category'])` | a "grouped" object (not printable yet) | sorted rows into groups, one per split + category combination |
| `.size()` | a **Series** with a two-level index | counted the rows in each group |
| `.unstack()` | a **DataFrame** | moved the inner index level (`category`) into columns, giving a readable table |

Each split has the same number of comfort and pain sentences. That **balance** means a coin-flip guess scores 50%, which is a useful reference later.

In [ ]:
# 🔧 Split into the three groups (the test set stays sealed: we won't touch test_df until Step 4)
build_df = pclab.split(sentences, 'build')
val_df = pclab.split(sentences, 'validation')
test_df = pclab.split(sentences, 'test')
display(build_df.head(6))

In the code, the build set's labels are called `y_train`. "Train" there means *building our ruler*. **The language model itself is never trained.**

## Part 2 · Where to take the snapshot

Every token has 1,536 numbers at every layer. That's a lot! We need to measure **the same spot** for every sentence, so we:

1. Put each sentence into the same **template**:
   > Description: The tea warmed her throat pleasantly.
   > The bodily sensation is
2. Take the snapshot at the **very last token** (" is").

**Why the last token?** The model reads left to right; each token can only "see" the tokens before it. By the last token it has seen the whole description, and it's the moment just before the model would name the sensation.

This setup is a **choice**, not the only option. A different template or token could give different results, which makes a good investigation (🌱 Step 7).

In [ ]:
# 🔧 Build one measurement prompt and look at its tokens
prompt = pclab.measurement_prompt(build_df['text'][0])
print(prompt)
print(repr(prompt))
ids = tokenizer(prompt)['input_ids']
display(pd.DataFrame({'position': range(len(ids)), 'token_piece': tokenizer.convert_ids_to_tokens(ids)}).tail(7))

`Ċ` is how this tokenizer displays the line break `\n` (you may see it glued to the full stop, as `.Ċ`). The **last row** is the token where we'll take our snapshot.

## Part 3 · Hooks: clip-on sensors

A **hook** is a small function you attach to one part of the model. Every time that part runs, PyTorch calls your function and hands it three things: **the part itself, what went in, and what came out**. It's like clipping a sensor onto a pipe: it can read what flows past without changing it.

Before building the real thing, let's attach a hook that just *reports what it receives*.

**✏️ Predict first:** what shape will the layer's output have for this prompt? (Hint: batch × tokens × numbers.)

In [ ]:
# 🔧 A hook that only prints what it receives
def report_hook(module, inputs, output):
    print('module :', type(module).__name__)
    print('inputs :', pclab.brief(inputs))
    print('output :', pclab.brief(output))

handle = LAYERS[0].register_forward_hook(report_hook)   # attach to the first layer
try:
    with torch.inference_mode():
        enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
        model.model(**enc, use_cache=False)
finally:
    handle.remove()                                     # always detach the sensor afterwards

### 📦 What came back?

The hook saw the layer's output: **(1 sentence, N tokens, 1,536 numbers)**. Depending on your version of the Transformers library, the output arrives either as a tensor directly or as a tuple whose first item is the tensor. Our code handles both.

> **🎛️ Why `handle.remove()` inside `finally:`?** A hook stays attached until removed. If we forgot, it would keep running on every future sentence. `try: ... finally:` guarantees the removal happens *even if an error occurs* in between.
>
> **🎛️ Why `model.model(...)` and not `model(...)`?** `model.model` is the **backbone**: all the layers, without the final `lm_head` that scores ~152,000 vocabulary entries. We only need the hidden numbers, so we skip that expensive last step.

### The real snapshot function

Now the function the whole lab relies on. It attaches one hook to **every** layer, runs the sentence once, and keeps only the **last token's** 1,536 numbers from each layer. Read it slowly, then use the walk-through below.

In [ ]:
# 🔧 extract_one: snapshot one sentence at every layer
@torch.inference_mode()
def extract_one(model, tokenizer, text):
    """Return the last-token numbers from every layer: shape (n_layers, hidden_size)."""
    layers = pclab.get_layers(model)
    collected = [None] * len(layers)          # one empty slot per layer
    handles = []

    def observer(index):                      # makes a hook that remembers which layer it's on
        def hook(module, inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output   # (batch, tokens, hidden)
            collected[index] = hidden[0, -1, :].detach().float().cpu().numpy().copy()
        return hook

    try:
        for index, block in enumerate(layers):
            handles.append(block.register_forward_hook(observer(index)))
        inputs = tokenizer(pclab.measurement_prompt(text), return_tensors='pt').to(DEVICE)
        model.model(**inputs, use_cache=False)
    finally:
        for handle in handles:
            handle.remove()

    return np.stack(collected)                # list of n_layers vectors → one (n_layers, hidden) array

one = extract_one(model, tokenizer, build_df['text'][0])
look(one, 'one sentence', dims=('layers', 'numbers'))

### 🔍 Code walk-through: `extract_one`

| Code | What it does | Why it's written this way |
|---|---|---|
| `@torch.inference_mode()` | runs the whole function in "use only" mode | no training bookkeeping: faster, less memory |
| `collected = [None] * len(layers)` | a list with one empty slot per layer | 🧱 a **list** because results arrive one at a time; slots let each hook write into *its own* place |
| `def observer(index): ... return hook` | a function that **makes** hook functions | each hook needs to remember its own layer number. Building it inside `observer(index)` "freezes" the right number into each hook. (Writing the hook directly inside the loop is a classic bug: every hook would end up with the *last* index.) |
| `output[0] if isinstance(output, tuple) else output` | gets the tensor out of the output | layers return a tuple in some library versions and a tensor in others |
| `hidden[0, -1, :]` | sentence 0, **last** token, all numbers | shape `(1, tokens, 1536)` → `(1536,)` |
| `.detach()` | stops tracking for training | not needed for our analysis |
| `.float()` | 32-bit numbers | consistent type for analysis |
| `.cpu()` | moves to main memory | NumPy can't read GPU memory |
| `.numpy()` | tensor → NumPy array | we analyse with NumPy |
| `.copy()` | an independent copy | the model may reuse its memory; a copy can't change behind our backs |
| `register_forward_hook(...)` | attaches a hook to a layer and returns a **handle** | the handle is the "remote control" we use to remove the hook later |
| `model.model(**inputs, use_cache=False)` | runs the backbone once | the hooks fire as the numbers flow through each layer; we don't need the return value |
| `np.stack(collected)` | turns a list of 28 arrays of shape `(1536,)` into **one** array `(28, 1536)` | 🧱 an array lets us do math across layers at once |

### 🛠️ Tinker: why not the *first* token?

Suppose we changed `hidden[0, -1, :]` to `hidden[0, 0, :]`, the **first** token ("Description").

**✏️ Predict first:** would the first-token snapshot differ between a pain sentence and its comfort partner?

In [ ]:
# 🔧 Compare FIRST-token numbers for a comfort sentence and its pain partner, at every stage
def all_stages(text):
    enc = tokenizer(pclab.measurement_prompt(text), return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        return model(**enc, output_hidden_states=True, use_cache=False).hidden_states

comfort_text, pain_text = build_df['text'][0], build_df['text'][1]
hs_comfort, hs_pain = all_stages(comfort_text), all_stages(pain_text)

def relative_difference(a, b):
    """How different two vectors are, as a fraction of the first one's size (0 = identical)."""
    return ((a - b).norm() / a.norm()).item()

first = [relative_difference(hs_comfort[k][0, 0], hs_pain[k][0, 0]) for k in range(len(hs_comfort))]
last = [relative_difference(hs_comfort[k][0, -1], hs_pain[k][0, -1]) for k in range(len(hs_comfort))]
print(f'FIRST token: largest difference across all stages = {max(first):.6f}')
print(f'LAST token : largest difference across all stages = {max(last):.6f}')

**The first token is identical** for both sentences at every layer (any difference is tiny rounding noise), while the last token clearly differs. Because the model only looks *backwards*, the first token has seen nothing but itself. So it can't carry any information about pain or comfort. That's the strongest argument for measuring at the **last** token.

### 🔭 Going deeper: does the hook agree with `output_hidden_states`?

In Step 1 we got hidden numbers with `output_hidden_states=True`. Entry `k + 1` there should equal our hook's snapshot of layer `k`. Let's check every layer.

In [ ]:
# 🔧 Compare our hook snapshots with hidden_states (remember: hidden_states[0] is the embedding)
from_hook = extract_one(model, tokenizer, comfort_text)
matches = [np.allclose(from_hook[k], hs_comfort[k + 1][0, -1].float().cpu().numpy(), rtol=1e-4, atol=1e-3)
           for k in range(N_LAYERS)]
print('Layers where they match:', [k + 1 for k, ok in enumerate(matches) if ok][:5], '...')
print('Layers where they differ:', [k + 1 for k, ok in enumerate(matches) if not ok])

They match at every layer **except the last**. Why? The Transformers library applies the model's final `norm` (from the printout in Step 1) to the last entry of `hidden_states`. Our hook reads each layer's raw output, *before* that rescaling. Neither is wrong; they're different measurement points, and good research says which one it used. Our results export records it as "layer output, last token, before final norm".

### From now on: the library version

`pclab.extract_one` is the same function, with one extra option for your own investigations (`reduce='mean'` averages all tokens instead of taking the last one). Let's confirm it gives the same numbers, then use it from here on.

In [ ]:
same = np.allclose(extract_one(model, tokenizer, comfort_text),
                   pclab.extract_one(model, tokenizer, comfort_text), atol=1e-5)
print('pclab.extract_one gives the same snapshot:', same)

## Part 4 · Collect snapshots for the build and validation sets

The next cell runs every build and validation sentence through the model, one at a time. It only *reads* numbers; nothing in the model changes. The test set stays sealed.

**✏️ Predict first:** what will the shape of `train_act` be? Write it as three numbers and say what each one means.

In [ ]:
# 🔧 Collect snapshots (usually under a minute on a GPU)
train_act = pclab.collect(model, tokenizer, build_df['text'], 'Build snapshots')
val_act = pclab.collect(model, tokenizer, val_df['text'], 'Validation snapshots')
y_train = build_df['label'].to_numpy()
y_val = val_df['label'].to_numpy()

look(train_act, 'train_act', dims=('sentences', 'layers', 'numbers'))
look(val_act, 'val_act', dims=('sentences', 'layers', 'numbers'))
look(y_train, 'y_train', dims=('sentences',))

### 📦 What came back?

`train_act` is a 3-D block: picture a **stack of 32 spreadsheets**, one per sentence, each with 28 rows (layers) and 1,536 columns (numbers). That's more than 1.3 million numbers, all from reading 32 short sentences.

`y_train` is a 1-D array of 0s and 1s in the **same order** as the sentences: `y_train[i]` is the label of `train_act[i]`.

> **🧱 Why one 3-D NumPy array instead of a list of lists, or a dictionary?** Because the questions we'll ask cut across it in different ways, and an array makes every cut one short expression:
>
> | Question | Expression | Shape |
> |---|---|---|
> | Everything about sentence 5 | `train_act[5]` | `(28, 1536)` |
> | Every sentence at layer 10 | `train_act[:, 9, :]` | `(32, 1536)` |
> | The average snapshot of all sentences | `train_act.mean(axis=0)` | `(28, 1536)` |
>
> Note `9` for "layer 10": **Python counts from 0.**

**Check your understanding:** if we added two more sentences, which of the three numbers in the shape would change? What if we switched to a model with 24 layers and 896 numbers per token?

## Part 5 · 🌱 A first look: are pain and comfort snapshots alike?

A simple way to compare two snapshots is **cosine similarity**: 1 means they point in exactly the same direction; 0 means unrelated. Let's compare each comfort sentence with **its own pain partner**, and with a **pain sentence from a different pair** (a stranger).

**✏️ Predict first:** will partners be more similar than strangers? Will either be anywhere near 0?

In [ ]:
# 🔧 Cosine similarity, layer by layer
# Build rows go comfort, pain, comfort, pain, ... (pair by pair), and masking keeps that order,
# so comfort[i] and pain[i] always come from the same pair.
comfort = train_act[y_train == 1]     # (16 sentences, layers, numbers)
pain = train_act[y_train == 0]        # (16 sentences, layers, numbers)
stranger = np.roll(pain, shift=1, axis=0)   # shift by one pair: every comfort sentence meets someone else's pain sentence

def cosine(a, b):
    return (a * b).sum(axis=-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1))

partners = cosine(comfort, pain).mean(axis=0)        # average over pairs → one value per layer
strangers = cosine(comfort, stranger).mean(axis=0)
look(partners, 'partners', dims=('layers',))

layer_numbers = np.arange(1, N_LAYERS + 1)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(layer_numbers, partners, 'o-', color=pclab.COLORS['accent'], label='comfort vs. its pain partner')
ax.plot(layer_numbers, strangers, 's--', color=pclab.COLORS['neutral'], label='comfort vs. a stranger\'s pain')
ax.set(xlabel='Layer', ylabel='Cosine similarity', title='How alike are the snapshots?')
ax.legend()
plt.show()

**What to notice:** the similarities are probably *high* everywhere, even for strangers. All our snapshots share the same template ("…The bodily sensation is") and the same position, and that shared part dominates the numbers. The pain–comfort difference, if there is one, is a *small* signal riding on a large shared background.

That's exactly why Step 3 doesn't compare raw snapshots. It **subtracts the averages** to cancel the shared background and isolate the difference.

---

## 🧭 Explain it

1. In one sentence each: what is a **hook**, and why do we remove it afterwards?
2. Why do we take the snapshot at the **last** token? What did the first-token experiment show?
3. Describe the shape of `train_act` in words, and say which dimension changes if you add sentences.
4. Why must the test set stay sealed until Step 4?

## 🌱 Curiosity branches

- **The template:** `pclab.collect(model, tokenizer, texts, prompt_fn=lambda t: 'Description: ' + t + '\nIt felt')` uses a different ending. Does the similarity picture change?
- **Where to look:** `pclab.collect(..., reduce='mean')` averages over every token instead of using the last. Is the pain–comfort difference easier or harder to see?
- **Partners vs. strangers:** which pair of sentences is *least* alike at the middle layer, and why might that be?

**Next:** [Step 3 · Finding a direction and choosing a layer](03_finding_a_direction.ipynb)